In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import os

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/title.basics.tsv", sep='\t')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
# Check missing values in each column

missing_values = pd.DataFrame({
    "Column": df.columns,
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df)) * 100
})

missing_values.sort_values(
    by="Missing Percentage",
    ascending=False
)

In [ ]:
df = df.drop('primaryTitle', axis=1)

In [ ]:
# Convert numerical columns

df["startYear"] = pd.to_numeric(
    df["startYear"],
    errors="coerce"
)

df["endYear"] = pd.to_numeric(
    df["endYear"],
    errors="coerce"
)

df["runtimeMinutes"] = pd.to_numeric(
    df["runtimeMinutes"],
    errors="coerce"
)

df["isAdult"] = pd.to_numeric(
    df["isAdult"],
    errors="coerce"
)


print("Data types converted successfully")

In [ ]:
# Keep realistic release years

df = df[
    (df["startYear"] >= 1800) &
    (df["startYear"] <= 2026)
]

print(
    "Records after year filtering:",
    f"{len(df):,}"
)

In [ ]:
df.head()

In [ ]:
# Remove unrealistic runtime values

df["runtimeMinutes"] = df["runtimeMinutes"].replace(
    np.nan,
    None
)

runtime_clean_df = df[
    (df["runtimeMinutes"].isna()) |
    (
        (df["runtimeMinutes"] > 0) &
        (df["runtimeMinutes"] < 1000)
    )
]


print(
    "Records after runtime cleaning:",
    f"{len(runtime_clean_df):,}"
)

In [ ]:
# Create cleaned dataframe

clean_df = runtime_clean_df.copy()

clean_df.head()

In [ ]:
clean_df["titleType"].value_counts()

In [ ]:
# Replace missing genres

clean_df["genres"] = clean_df["genres"].fillna(
    "Unknown"
)


# Split genres

clean_df["genre_list"] = clean_df["genres"].apply(
    lambda x: x.split(",")
)


clean_df[
    [

        "genres",
        "genre_list"
    ]
].head()

In [ ]:
# Create decade column

clean_df["decade"] = (
    clean_df["startYear"] // 10 * 10
)

clean_df[
    [
        "startYear",
        "decade"
    ]
].head()

In [ ]:
clean_df["adult_label"] = clean_df[
    "isAdult"
].map(
    {
        0: "Non-Adult",
        1: "Adult"
    }
)


clean_df[
    [
        "isAdult",
        "adult_label"
    ]
].head()

In [ ]:
print("Final Clean Dataset Information")
print("--------------------------------")

print(
    f"Rows: {clean_df.shape[0]:,}"
)

print(
    f"Columns: {clean_df.shape[1]}"
)

print(
    "\nColumns Available:"
)

for col in clean_df.columns:
    print("-", col)

## Title Type Distribution

In [ ]:
title_type_count = (
    clean_df["titleType"]
    .value_counts()
    .reset_index()
)

title_type_count.columns = [
    "Title Type",
    "Count"
]


fig = px.bar(
    title_type_count,
    x="Title Type",
    y="Count",
    title="Distribution of IMDb Title Types",
    text="Count"
)

fig.update_layout(
    xaxis_tickangle=-45
)

fig.show()



## Adult vs Non-Adult Content

In [ ]:
adult_count = (
    clean_df["adult_label"]
    .value_counts()
    .reset_index()
)

adult_count.columns = [
    "Category",
    "Count"
]


fig = px.pie(
    adult_count,
    names="Category",
    values="Count",
    title="Adult vs Non-Adult Titles"
)

fig.show()



## Titles Released Over Time

In [ ]:
year_count = (
    clean_df["startYear"]
    .value_counts()
    .sort_index()
    .reset_index()
)

year_count.columns = [
    "Year",
    "Number of Titles"
]


fig = px.line(
    year_count,
    x="Year",
    y="Number of Titles",
    title="IMDb Titles Released Over Time"
)

fig.show()



## Titles by Decade

In [ ]:
decade_count = (
    clean_df["decade"]
    .value_counts()
    .sort_index()
    .reset_index()
)

decade_count.columns = [
    "Decade",
    "Number of Titles"
]


fig = px.bar(
    decade_count,
    x="Decade",
    y="Number of Titles",
    title="IMDb Titles by Decade"
)

fig.show()



## Runtime Distribution

In [ ]:
runtime_df = clean_df.dropna(
    subset=["runtimeMinutes"]
)


fig = px.histogram(
    runtime_df,
    x="runtimeMinutes",
    nbins=100,
    title="Distribution of Runtime (Minutes)"
)

fig.update_layout(
    xaxis_title="Runtime Minutes",
    yaxis_title="Number of Titles"
)

fig.show()




## Average Runtime Comparison

In [ ]:
runtime_by_type = (
    runtime_df
    .groupby("titleType")
    ["runtimeMinutes"]
    .mean()
    .reset_index()
)


runtime_by_type.columns = [
    "Title Type",
    "Average Runtime"
]


fig = px.bar(
    runtime_by_type,
    x="Title Type",
    y="Average Runtime",
    title="Average Runtime by Title Type"
)

fig.update_layout(
    xaxis_tickangle=-45
)

fig.show()



## Runtime Variation

In [ ]:
runtime_box = runtime_df[
    runtime_df["runtimeMinutes"] < 300
]


fig = px.box(
    runtime_box,
    x="titleType",
    y="runtimeMinutes",
    title="Runtime Distribution by Title Type"
)

fig.update_layout(
    xaxis_tickangle=-45
)

fig.show()




## Extract Individual Genres

In [ ]:
genre_df = (
    clean_df["genre_list"]
    .explode()
    .value_counts()
    .reset_index()
)

genre_df.columns = [
    "Genre",
    "Count"
]


genre_df.head()

## Top 20 Genres

In [ ]:
top_genres = genre_df.head(20)


fig = px.bar(
    top_genres,
    x="Genre",
    y="Count",
    title="Top 20 IMDb Genres",
    text="Count"
)

fig.update_layout(
    xaxis_tickangle=-45
)

fig.show()



## Interactive Genre Treemap

In [ ]:
fig = px.treemap(
    top_genres,
    path=["Genre"],
    values="Count",
    title="IMDb Genre Distribution Treemap"
)

fig.show()


